<a href="https://colab.research.google.com/github/dudugan/miniproject-aug9/blob/main/miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Setup

In [ ]:
#gpu check!
nvidia-smi

In [ ]:
!pip install unsloth
!pip install anthropic
!pip install httpx nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 

In [ ]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
import anthropic
client = anthropic.Anthropic()
response = client.messages.create(
    model = "claude-sonnet-4-6",
    max_tokens = 50,
    messages = [{"role": "user", "content": "Print 'Nom Nom' and nothing else."}]
)
print(response.content[0].text)

```python
print('Nom Nom')
```

Output:
```
Nom Nom
```


In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Question Database

In [ ]:
import json
import re

def generate_contexts_batch(n, theme_hint, client, model="claude-sonnet-4-6"):
    """Ask Claude for n (context, advice) pairs as JSON."""
    prompt = f"""Generate {n} realistic examples of a coding assistant interaction.

Each example has:
- "context": a short (1-3 sentence) description of a coding situation a user is asking about
- "advice": a piece of substantive, correct advice an LLM might give in response (2-4 sentences)

Focus this batch loosely on: {theme_hint}

Vary the languages, frameworks, and situation types (debugging, architecture, performance, security, code review, etc).

Respond with ONLY a JSON array, no preamble, no markdown fences. Format:
[{{"context": "...", "advice": "..."}}, ...]
"""
    response = client.messages.create(
        model=model,
        max_tokens=8000,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.content[0].text.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text.strip())

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [!] JSON parse failed ({e}). Attempting repair...")
        # truncating to last complete thing
        last_close = text.rfind("},")
        if last_close != -1:
            repaired = text[:last_close+1] + "]"
            try:
                result = json.loads(repaired)
                print(f"  [!] Repaired: recovered {len(result)}/{n} examples")
                return result
            except json.JSONDecodeError:
                pass
        print(f"  [!] Repair failed. Raw text saved to debug for inspection.")
        return []  # empty batch to avoid crash

In [ ]:
themes = [
    "debugging and fixing bugs",
    "software architecture and design decisions",
    "performance optimization",
    "security, testing, and code review practices",
]

all_contexts = []
for theme in themes:
    batch = generate_contexts_batch(25, theme, client)
    all_contexts.extend(batch)
    print(f"Got {len(batch)} examples for theme: {theme}")

print(f"\nTotal: {len(all_contexts)} contexts")

Got 25 examples for theme: debugging and fixing bugs
Got 25 examples for theme: software architecture and design decisions
Got 25 examples for theme: performance optimization
Got 25 examples for theme: security, testing, and code review practices

Total: 100 contexts


In [ ]:
for ex in all_contexts[:2]:
    print(ex)
    print()

with open("coding_contexts.json", "w") as f:
    json.dump(all_contexts, f, indent=2)

print(f"Saved {len(all_contexts)} contexts to coding_contexts.json")

{'context': "A Python developer is getting a 'KeyError' when trying to access a dictionary value, but they're sure the key exists.", 'advice': "KeyErrors often occur due to subtle differences like trailing whitespace, different casing, or Unicode characters that look identical but aren't. Use `print(repr(key))` to inspect the exact representation of both the key you're inserting and the one you're looking up. As a safer alternative, use `dict.get(key, default)` to avoid the exception and return a fallback value instead."}

{'context': 'A JavaScript developer notices their async function is not awaiting a Promise correctly, and the function returns undefined instead of the expected data.', 'advice': "This typically happens when you forget to use the `await` keyword before a Promise-returning function call, or when the outer function is not marked as `async`. Make sure both the function definition includes `async` and every Promise call inside it has `await` in front of it. Also check th

In [ ]:
!git clone https://github.com/dudugan/miniproject-aug9.git
!cp coding_contexts.json miniproject-aug9/
!cd miniproject-aug9 && git add coding_contexts.json && git commit -m "Add generated contexts" && git config --global user.email "dvzar27@gmail.com" && git config --global user.name "dudugan"
!cd miniproject-aug9 && git push

Cloning into 'miniproject-aug9'...
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@1fa63a204cbb.(none)')
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/dudugan/miniproject-aug9.git'
